## Silver Layer — Activity Weather
Cleans and standardizes weather data from Garmin API.

**Transformations:**
* Temperature: Fahrenheit → Celsius
* Wind speed: mph → km/h
* Extract weather condition from JSON
* Parse timestamp
* Filter: Only activities with actual weather data (temp IS NOT NULL)

In [0]:
%sql
CREATE OR REPLACE TABLE garmin_lakehouse.silver.activity_weather AS
SELECT
    CAST(activityId AS STRING) as activity_id,
    
    -- Temperature (convert from Fahrenheit to Celsius)
    ROUND((temp - 32) * 5.0 / 9.0, 1) as temperature_c,
    ROUND((apparentTemp - 32) * 5.0 / 9.0, 1) as feels_like_c,
    
    
    -- Wind (convert mph to km/h)
    ROUND(windSpeed * 1.60934, 1) as wind_speed_kmh,
    
    -- Weather condition (extract from JSON)
    get_json_object(weatherTypeDTO, '$.desc') as weather_condition,
    
    -- Metadata
    CURRENT_TIMESTAMP() as loaded_at

FROM garmin_lakehouse.raw.activity_weather
WHERE temp IS NOT NULL  -- Only include activities with actual weather data